# 02 — Random Walk → Brownian Motion → Geometric Brownian Motion

This notebook builds up from a simple discrete random walk, through the scaling limit that connects it to standard Brownian motion (Donsker's theorem), to Geometric Brownian Motion (GBM) — the classic stock price model underlying Black-Scholes. We finish by comparing simulated GBM log-returns against real market data to see where the model's lognormal assumption breaks down (fat tails).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from quant_sims.stochastic_processes import RandomWalk, BrownianMotion, GeometricBrownianMotion
from quant_sims.utils.plotting import plot_paths, plot_terminal_distribution_vs_theory

%matplotlib inline

## 1. Simple Random Walk

S_n = sum of n independent +-1 steps. For an unbiased walk, E[S_n] = 0 and Var(S_n) = n.

In [ ]:
walk = RandomWalk(p_up=0.5, seed=42)
paths = walk.simulate_paths(n_steps=500, n_paths=300)
time_grid = np.arange(paths.shape[1])

plot_paths(time_grid, paths, title="Unbiased random walk (500 steps, 300 paths)", ylabel="S_n")
plt.show()

final_values = paths[:, -1]
print(f"Empirical mean of S_500: {np.mean(final_values):.2f} (theory: 0)")
print(f"Empirical variance of S_500: {np.var(final_values):.1f} (theory: 500)")

## 2. Scaling limit: random walk → Brownian motion

By Donsker's theorem, rescaling S_floor(n*t) by 1/sqrt(n) and letting n grow converges to a standard Wiener process. We visualize this by comparing rescaled random walks at increasing n against a directly simulated Brownian path — as n grows, the walk looks visually indistinguishable from continuous Brownian motion.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)

for ax, n in zip(axes, [10, 100, 1000, 10000]):
    w = RandomWalk(seed=1)
    t, p = w.scaling_limit_paths(n_steps=n, n_paths=1, T=1.0)
    ax.plot(t, p[0], color="steelblue", linewidth=1)
    ax.set_title(f"n = {n} steps")
    ax.set_xlabel("t")

axes[0].set_ylabel("W_n(t)")
fig.suptitle("Rescaled random walk converging to a Brownian path as n grows")
plt.tight_layout()
plt.show()

## 3. Brownian Motion (Wiener process)

Directly simulating W(t) with drift mu and volatility sigma, using exact Gaussian increments: W(t_{k+1}) = W(t_k) + sqrt(dt) * Z.

In [ ]:
mu, sigma = 0.0, 1.0
T, n_steps, n_paths = 2.0, 400, 300

bm = BrownianMotion(mu=mu, sigma=sigma, seed=42)
t, paths = bm.simulate_paths(T=T, n_steps=n_steps, n_paths=n_paths)

plot_paths(t, paths, title="Standard Brownian motion paths", ylabel="W(t)")
plt.show()

final_values = paths[:, -1]
theory_mean = BrownianMotion.theoretical_mean(np.array([T]), mu)[0]
theory_std = BrownianMotion.theoretical_std(np.array([T]), sigma)[0]
print(f"Empirical mean of W({T}): {np.mean(final_values):.3f} (theory: {theory_mean:.3f})")
print(f"Empirical std of W({T}):  {np.std(final_values):.3f} (theory: {theory_std:.3f})")

## 4. Geometric Brownian Motion (GBM)

S_t = S_0 * exp[(mu - sigma^2/2) t + sigma W(t)], simulated exactly via the closed-form solution. This is the classic model for stock prices — always positive, lognormally distributed.

In [ ]:
S0 = 100.0
mu, sigma = 0.08, 0.25   # 8% annual drift, 25% annual volatility
T, n_steps, n_paths = 1.0, 252, 500   # 1 year, daily steps

gbm = GeometricBrownianMotion(mu=mu, sigma=sigma, seed=7)
t, paths = gbm.simulate_paths(S0=S0, T=T, n_steps=n_steps, n_paths=n_paths)

plot_paths(t, paths, title="GBM simulated stock price paths", ylabel="S(t)")
plt.show()

In [ ]:
final_values = paths[:, -1]

# Overlay the theoretical lognormal density on the empirical terminal distribution
log_mean = np.log(S0) + (mu - 0.5 * sigma**2) * T
log_std = sigma * np.sqrt(T)
lognormal_pdf = lambda x: stats.lognorm.pdf(x, s=log_std, scale=np.exp(log_mean))

plot_terminal_distribution_vs_theory(final_values, theoretical_pdf_fn=lognormal_pdf, title="GBM terminal price distribution vs. theoretical lognormal")
plt.show()

print(f"Empirical mean S(T): {np.mean(final_values):.2f} (theory: {gbm.theoretical_mean(S0, np.array([T]))[0]:.2f})")

## 5. Reality check: GBM vs. real market log-returns

GBM assumes log-returns are normally distributed. Real markets tend to show fat tails (excess kurtosis) — extreme moves happen more often than a normal distribution predicts. We fetch real S&P 500 data (requires internet access; skipped gracefully if unavailable) and compare its log-return distribution to a fitted GBM.

In [ ]:
try:
    import yfinance as yf

    spx = yf.download("^GSPC", period="5y", interval="1d", progress=False)
    prices = spx["Close"].dropna().values.flatten()
    have_real_data = len(prices) > 100
except Exception as e:
    print(f"Could not fetch real market data ({e}). Skipping this comparison — install `yfinance` and ensure internet access to run it.")
    have_real_data = False

In [ ]:
if have_real_data:
    dt = 1 / 252  # daily steps, annualized
    mu_hat, sigma_hat = GeometricBrownianMotion.fit_mu_sigma(prices, dt=dt)
    print(f"Fitted from real S&P 500 data: mu = {mu_hat:.3f}, sigma = {sigma_hat:.3f}")

    real_log_returns = np.diff(np.log(prices))

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(real_log_returns, bins=80, density=True, alpha=0.6, color="steelblue", label="Real S&P 500 daily log-returns")

    x = np.linspace(real_log_returns.min(), real_log_returns.max(), 400)
    normal_fit = stats.norm.pdf(x, loc=np.mean(real_log_returns), scale=np.std(real_log_returns))
    ax.plot(x, normal_fit, color="darkorange", linewidth=2, label="Normal fit (GBM assumption)")

    ax.set_title("Real log-returns vs. GBM's normal assumption")
    ax.set_xlabel("Daily log-return")
    ax.set_ylabel("Density")
    ax.legend()
    plt.show()

    excess_kurtosis = stats.kurtosis(real_log_returns)  # 0 for a true normal distribution
    print(f"Excess kurtosis of real log-returns: {excess_kurtosis:.2f} (0 would match GBM's normal assumption)")
    print("Positive excess kurtosis = fatter tails than GBM predicts: extreme moves are more common than the model assumes.")

## Takeaways

- A simple random walk, properly rescaled, converges to Brownian motion — this is Donsker's theorem in action, and it's the same coin-flip mechanics from Module 1, viewed as an additive rather than multiplicative process.
- Brownian motion can be simulated *exactly* on any time grid because its increments are, by definition, i.i.d. Gaussian — no discretization error.
- GBM applies Ito's lemma to log(S) to get a closed-form solution, guaranteeing S_t stays positive and is lognormally distributed — this is the foundation of Black-Scholes.
- Real market log-returns show fatter tails than GBM predicts (positive excess kurtosis). This is GBM's best-known limitation and motivates later modules: jump diffusion (Merton) and stochastic volatility (Heston).